# Análisis de Errores del Sistema

**Objetivo:** Identificar, clasificar y analizar los errores producidos por el sistema de análisis
**Duración estimada:** 35 minutos

---

## Contenido

1. [Setup](#setup)
2. [Tipos de Errores del Sistema](#tipos-de-errores)
3. [Errores de Parsing](#errores-de-parsing)
4. [Errores de Análisis de Complejidad](#errores-de-analisis)
5. [Errores de Detección de Patrones](#errores-de-patrones)
6. [Análisis de Casos Límite](#casos-limite)
7. [Clasificación y Frecuencia de Errores](#clasificacion)
8. [Recomendaciones](#recomendaciones)

---

## 1. Setup

In [ ]:
import sys
import traceback
from collections import Counter, defaultdict

sys.path.insert(0, '../..')

from app.core.parser import parse_pseudocode
from app.core.analyzer import AnalyzerEngine
from app.core.patterns import PatternDetector
from app.core.data_structures import StructureIdentifier
from app.core.exceptions import ParserException

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("Setup completado")

---

## 2. Tipos de Errores del Sistema

Los errores del sistema se clasifican en cuatro categorías principales:

| Categoría | Descripción | Impacto |
|-----------|-------------|---------|
| Parsing | El pseudocódigo no puede ser interpretado | Alto |
| Análisis | El motor no puede calcular la complejidad | Medio |
| Detección | El patrón o estructura no es reconocido | Bajo |
| Timeout | El procesamiento supera el tiempo límite | Variable |

In [ ]:
# Registro centralizado de errores para este análisis
registro_errores = defaultdict(list)

def registrar_error(categoria, nombre, descripcion, codigo_fuente=None):
    """Registra un error para análisis posterior."""
    registro_errores[categoria].append({
        "nombre": nombre,
        "descripcion": descripcion,
        "codigo_fuente": codigo_fuente
    })
    print(f"[{categoria}] {nombre}: {descripcion[:60]}")

print("Sistema de registro de errores inicializado")

---

## 3. Errores de Parsing

Los errores de parsing ocurren cuando el pseudocódigo tiene sintaxis que la gramática Lark no puede procesar.
Es la categoría de error más crítica porque bloquea todo el pipeline.

### 3.1 Casos que Deben Fallar (Sintaxis Inválida)

In [ ]:
CODIGOS_INVALIDOS = [
    ("Falta BEGIN", """
algorithm test(n)
    x <- 5
end
"""),
    ("Falta END", """
algorithm test(n)
begin
    x <- 5
"""),
    ("Operador de asignación incorrecto", """
algorithm test(n)
begin
    x = 5
end
"""),
    ("Palabra clave faltante THEN", """
algorithm test(n)
begin
    if (n > 0)
        x <- 1
    end
end
"""),
    ("Paréntesis no balanceados en while", """
algorithm test(n)
begin
    while (n > 0 do
        n <- n - 1
    end
end
"""),
]

print("PRUEBAS DE SINTAXIS INVÁLIDA (todos deben fallar):")
for nombre, codigo in CODIGOS_INVALIDOS:
    try:
        ast = parse_pseudocode(codigo)
        print(f"  PROBLEMA - {nombre}: No generó error cuando debería")
        registrar_error("Parsing", nombre, "Código inválido aceptado incorrectamente", codigo)
    except (ParserException, Exception) as e:
        print(f"  CORRECTO  - {nombre}: Error detectado correctamente")

### 3.2 Casos Límite de Parsing

In [ ]:
CASOS_LIMITE_PARSING = [
    ("Algoritmo vacío", """
algorithm vacio()
begin
end
"""),
    ("Nombre con underscore", """
algorithm mi_algoritmo(n)
begin
    x <- n
end
"""),
    ("Anidamiento profundo (nivel 5)", """
algorithm anidado(n)
begin
    for i <- 1 to n do
        for j <- 1 to n do
            for k <- 1 to n do
                for l <- 1 to n do
                    for m <- 1 to n do
                        x <- i + j + k + l + m
                    end
                end
            end
        end
    end
end
"""),
    ("Expresión matemática compleja", """
algorithm formula(a, b, c, d)
begin
    resultado <- floor((a + b * c - d) / (a * b + 1))
end
"""),
    ("Array multidimensional", """
algorithm matriz(M[][], n, m)
begin
    for i <- 1 to n do
        for j <- 1 to m do
            M[i][j] <- 0
        end
    end
end
"""),
]

print("CASOS LÍMITE DE PARSING:")
errores_parsing = 0
for nombre, codigo in CASOS_LIMITE_PARSING:
    try:
        ast = parse_pseudocode(codigo)
        stmts = len(ast.algorithm.body.statements)
        print(f"  OK  - {nombre}: {stmts} statement(s)")
    except Exception as e:
        errores_parsing += 1
        tipo_error = type(e).__name__
        registrar_error("Parsing", nombre, f"{tipo_error}: {str(e)[:50]}", codigo)
        print(f"  FAIL - {nombre}: {tipo_error}")

print(f"\nErrores en casos límite de parsing: {errores_parsing}/{len(CASOS_LIMITE_PARSING)}")

---

## 4. Errores de Análisis de Complejidad

In [ ]:
engine = AnalyzerEngine()

CASOS_ANALISIS_DIFICILES = [
    ("Función sin retorno claro", """
algorithm sideEffect(A[], n)
begin
    for i <- 1 to n do
        A[i] <- A[i] * 2
    end
end
"""),
    ("Doble recursión con parámetro dividido", """
algorithm divideRecurse(n)
begin
    if (n <= 1) then
        return 1
    end
    call divideRecurse(n / 3)
    call divideRecurse(n / 3)
    call divideRecurse(n / 3)
end
"""),
    ("While con condición compuesta", """
algorithm buscarDual(A[], n, x, y)
begin
    i <- 1
    j <- n
    while (i < j and A[i] != x) do
        if (A[i] > y) then
            j <- j - 1
        else
            i <- i + 1
        end
    end
end
"""),
]

print("CASOS DIFÍCILES PARA EL ANALIZADOR:")
errores_analisis = 0
for nombre, codigo in CASOS_ANALISIS_DIFICILES:
    try:
        ast = parse_pseudocode(codigo)
        analysis = engine.analyze(ast)
        big_o = getattr(analysis, 'big_o', None) or getattr(analysis, 'time_complexity', {}).get('big_o', 'N/A')
        print(f"  OK  - {nombre}: BigO={big_o}")
    except Exception as e:
        errores_analisis += 1
        tipo_error = type(e).__name__
        registrar_error("Análisis", nombre, f"{tipo_error}: {str(e)[:50]}", codigo)
        print(f"  FAIL - {nombre}: {tipo_error}: {str(e)[:50]}")

print(f"\nErrores en análisis complejo: {errores_analisis}/{len(CASOS_ANALISIS_DIFICILES)}")

---

## 5. Errores de Detección de Patrones

In [ ]:
detector = PatternDetector()

CASOS_PATRONES_AMBIGUOS = [
    ("Mezcla de divide y vence con DP", """
algorithm optimalSearch(A[], n)
begin
    memo <- 0
    if (n <= 1) then
        return A[1]
    end
    mid <- floor(n / 2)
    call optimalSearch(A, mid)
    call optimalSearch(A, n - mid)
    memo <- memo + 1
end
"""),
    ("Greedy con backtracking limitado", """
algorithm greedyWithCheck(n)
begin
    resultado <- 0
    i <- n
    while (i > 0) do
        if (i > resultado) then
            resultado <- i
        end
        i <- i - 1
    end
    return resultado
end
"""),
    ("Algoritmo sin patrón claro", """
algorithm procesarDatos(x, y, z)
begin
    a <- x + y
    b <- y * z
    c <- a - b
    return c
end
"""),
]

print("CASOS AMBIGUOS PARA EL DETECTOR DE PATRONES:")
for nombre, codigo in CASOS_PATRONES_AMBIGUOS:
    try:
        ast = parse_pseudocode(codigo)
        detection = detector.detect(ast)
        patron = getattr(detection, 'primary_pattern', None)
        confianza = getattr(detection, 'primary_confidence', 0.0)
        print(f"  OK  - {nombre}: patron={patron}, confianza={confianza:.2f}")
        if confianza < 0.4:
            registrar_error("Patrones", nombre, f"Baja confianza ({confianza:.2f}) en detección", codigo)
    except Exception as e:
        registrar_error("Patrones", nombre, f"{type(e).__name__}: {str(e)[:50]}", codigo)
        print(f"  FAIL - {nombre}: {type(e).__name__}")

---

## 6. Análisis de Casos Límite

In [ ]:
CASOS_EXTREMOS = [
    ("Algoritmo de una línea", """
algorithm identidad(x)
begin
    return x
end
"""),
    ("Sin parámetros", """
algorithm constante()
begin
    return 42
end
"""),
    ("Operaciones en cadena", """
algorithm cadenaOp(n)
begin
    a <- n + 1
    b <- a + 1
    c <- b + 1
    d <- c + 1
    e <- d + 1
    return e
end
"""),
]

print("CASOS EXTREMOS Y LÍMITES:")
resultados_extremos = []
for nombre, codigo in CASOS_EXTREMOS:
    resultado = {"nombre": nombre, "parsing": False, "analisis": False, "patrones": False}
    
    try:
        ast = parse_pseudocode(codigo)
        resultado["parsing"] = True
        
        analysis = engine.analyze(ast)
        resultado["analisis"] = True
        
        detection = detector.detect(ast)
        resultado["patrones"] = True
        
    except Exception as e:
        registrar_error("CasoExtremo", nombre, f"{type(e).__name__}: {str(e)[:50]}", codigo)
    
    resultados_extremos.append(resultado)
    estado = "OK" if all([resultado["parsing"], resultado["analisis"], resultado["patrones"]]) else "PARCIAL"
    print(f"  {estado} - {nombre}: parsing={resultado['parsing']}, analisis={resultado['analisis']}, patrones={resultado['patrones']}")

---

## 7. Clasificación y Frecuencia de Errores

In [ ]:
# Contar errores por categoría
conteo_categorias = Counter(cat for cat in registro_errores)
total_errores = sum(len(v) for v in registro_errores.values())

print(f"\nTOTAL DE ERRORES REGISTRADOS: {total_errores}")
for cat, errores in registro_errores.items():
    print(f"  {cat}: {len(errores)} error(s)")
    for e in errores:
        print(f"    - {e['nombre']}: {e['descripcion'][:50]}")

# Visualización
if total_errores > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    categorias = list(registro_errores.keys())
    cantidades = [len(registro_errores[c]) for c in categorias]
    colores = ["#C44E52", "#DD8452", "#4C72B0", "#55A868"][:len(categorias)]
    
    bars = ax.barh(categorias, cantidades, color=colores, alpha=0.85)
    ax.set_xlabel("Número de Errores")
    ax.set_title("Distribución de Errores por Categoría")
    
    for bar, val in zip(bars, cantidades):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig("error_analysis.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("No se registraron errores en esta ejecución.")

---

## 8. Recomendaciones

In [ ]:
print("RECOMENDACIONES BASADAS EN EL ANÁLISIS DE ERRORES")

recomendaciones = {
    "Parsing": [
        "Mejorar mensajes de error indicando la línea y columna exacta del fallo",
        "Ampliar la gramática para soportar más variantes de sintaxis",
        "Agregar recuperación de errores para continuar parseo parcial",
    ],
    "Análisis": [
        "Manejar casos donde el ciclo while tiene condición no analizable simbólicamente",
        "Agregar soporte para recursión múltiple con diferente factor de división",
        "Documentar los casos donde el análisis retorna estimación conservadora",
    ],
    "Patrones": [
        "Establecer umbral mínimo de confianza configurable (actualmente implícito)",
        "Mejorar la detección de patrones mixtos (DP + divide y vencerás)",
        "Agregar patrón 'SIN_PATRÓN_CLARO' para algoritmos de propósito general",
    ],
}

for categoria, items in recomendaciones.items():
    errores_en_cat = len(registro_errores.get(categoria, []))
    prioridad = "ALTA" if errores_en_cat > 2 else "MEDIA" if errores_en_cat > 0 else "BAJA"
    print(f"\n{categoria} (Prioridad: {prioridad}, Errores: {errores_en_cat}):")
    for item in items:
        print(f"  - {item}")

print("\n" + "=" * 60)

---

## Proximos Pasos

- **performance_metrics.ipynb**: Analizar el rendimiento del sistema
- **accuracy_evaluation.ipynb**: Evaluar la precisión con más casos de referencia